In [ ]:
import numpy as np

from qarp.blocks import QSVTBlock, QSPAngleFinder, BlockEncodingBlock
import qarpx as qx

In [ ]:
# The Quantum Singular Value Transformation computes a matrix A under a specific and suitable polynomial transformation

A = np.array([[0.5, 0.3], [0.3, -0.5]])

# The matrix must be properly normalized, we can take it into account explicitly, but it's also done internally 
lambda_factor = BlockEncodingBlock(A).lambda_factor()
A = A / lambda_factor

# Polynomial transformation that will transform A to A^2 - 1. Bear in mind that not all transformations are valid for QSVT, and the pipeline does not check it at the moment
P_poly = [-1,0,1]  # x^2 - 1

# We find the transformation by means of appropriate phase rotations. We find the optimal angles for that
optimal_angles = QSPAngleFinder(P_poly).QSVT()

# Build the QSVT block, which you can append to a circuit. The resulting transformed matrix will be encoded in the first qubits
qsvt = QSVTBlock(A, optimal_angles).build()

# Let's check we obtain the transformed A embedded in the QSVT unitary (careful: it will be encoded only in the REAL part)
qsvt_mat = np.array(qx.QarpSimulator().unitary_matrix(qsvt.flatten(), qsvt.n_qubits))

# The block-encoded p(A) lives on the |0...0>_ancilla subspace. In qarpx's LSB
# convention the ancilla register is the *low* qubits, so those subspace indices
# are strided (0, N_anc, 2*N_anc, ...), NOT the literal top-left corner.
Arows, Acols = A.shape
n_anc = qsvt.n_qubits - int(np.ceil(np.log2(Arows)))
N_anc = 2 ** n_anc
idx = np.arange(Arows) * N_anc

# Check the block applies the transformation accordingly
np.linalg.norm( np.real(qsvt_mat[np.ix_(idx, idx)]) - (A @ A - np.eye(2)) )

In [ ]:
# The QSVT block is a concatenation of the ProjectedControlled blocks with optimal angles and the BlockEncoding of the matrix
qsvt.plot(spacing=0.5)